# Lesson 4.2 - Resource Provisioning and Automation

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Create a simple configuration file that describes a cloud environment
- Write a runbook-style deployment script that reads configuration values
- Keep deployment logic predictable and reviewable

**Project file(s) produced:** `config/environments.json` and `runbooks/Deploy-DevEnvironment.ps1`

## Concepts

Production automation usually begins with a config file. That gives you one place to store names, locations, and tags instead of hard-coding them in several scripts.

A deployment script should read configuration, make the plan obvious, and keep anything risky behind a deliberate action.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-4-cloud-devops')
config_dir = lesson_dir / 'config'
runbook_dir = lesson_dir / 'runbooks'
artifact_dir = lesson_dir / 'artifacts'
config_dir.mkdir(parents=True, exist_ok=True)
runbook_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Store environment settings in JSON

This file gives the deployment script a single source of truth. Later you can replace the local file with Key Vault, pipeline variables, or another secure source.

In [ ]:
config_path = config_dir / 'environments.json'
config = {
    'dev': {
        'resourceGroupName': 'rg-powershell-dev',
        'location': 'eastus',
        'tags': { 'Project': 'PowerShell Study Plan', 'Environment': 'dev' }
    }
}
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

## Code-Along: Step 2 - Write a runbook-style deployment script

This step creates a script that reads the JSON config and prints a clear deployment plan. In a real environment you would replace the comments with the actual Azure commands.

In [ ]:
runbook_path = runbook_dir / 'Deploy-DevEnvironment.ps1'
runbook_content = textwrap.dedent(r'''
param(
    [Parameter(Mandatory)]
    [string]$EnvironmentName
)

$configPath = Join-Path $PSScriptRoot '..\config\environments.json'
$config = Get-Content -Path $configPath | ConvertFrom-Json
$envConfig = $config.$EnvironmentName

# TODO: replace the preview output with real Azure provisioning commands
[pscustomobject]@{
    ResourceGroup = $envConfig.resourceGroupName
    Location      = $envConfig.location
    TagCount      = $envConfig.tags.PSObject.Properties.Count
} | Format-List
''')
runbook_path.write_text(runbook_content, encoding='utf-8')
print(runbook_path.read_text(encoding='utf-8'))

## Exercise

Add a second environment entry to the config file so the runbook can target staging as well as dev.

**Your task:**
1. Add a staging object to the JSON file.
2. Give it a different resource group name.
3. Confirm the runbook can still read the config file.

**Expected result:** a config-driven deployment pattern that scales to multiple environments.

In [ ]:
# TODO: add a staging environment to the config file
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: add a second environment"
''')
run_powershell(exercise_script, 'lesson-4.2-exercise.ps1')

## Key Takeaways

- A config file keeps deployment inputs organized.
- Runbook-style scripts should be explicit about their inputs and outputs.
- The same pattern can support dev, staging, and production with different values.

---

**Next:** [Lesson 4.3 - CI/CD and Secrets](./lesson-4.3-ci-cd-secrets.ipynb)